# 🎃 ZapalloAI — Notebook 02: Preprocesamiento y Unificación

**Universidad de las Fuerzas Armadas ESPE**

Este notebook:
1. Unifica las clases de los dos datasets con el mapeo definido
2. Elimina duplicados
3. Realiza un split estratificado **70% train / 15% val / 15% test**
4. Aplica augmentations offline con Albumentations
5. Genera estadísticas del dataset final

## Mapeo de clases
| Clase objetivo | Sweet Pumpkin | Cucurbit Leaf |
|---|---|---|
| `healthy` | Healthy | healthy |
| `downy_mildew` | Powdery Mildew | downy_mildew |
| `leaf_curl` | Leaf Curl | leaf_curl_virus |
| `mosaic_virus` | Mosaic Virus | mosaic |
| `red_beetle` | Red Beetle | *(no existe)* |

## Modos de ejecución
- **Local**: coloca los datasets en `model/data/raw/` y ejecuta con Jupyter o VS Code
- **Google Colab**: monta tu Drive y ajusta `COLAB_MODE = True`

In [ ]:
# ── 0. Instalar dependencias ──────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'albumentations', 'imagehash', 'opencv-python-headless',
                'Pillow', 'matplotlib', 'pandas'], check=False)
print('✅ Dependencias instaladas')

In [ ]:
# ── 1. Configuración de rutas (LOCAL o COLAB) ─────────────────────
import os, sys, shutil, random
import numpy as np
from pathlib import Path
from PIL import Image
import imagehash

# ─── CAMBIAR AQUÍ SEGÚN DÓNDE EJECUTAS ───────────────────────────
COLAB_MODE = False   # True  → Google Colab + Drive
                     # False → Local (VS Code / Jupyter)
# ──────────────────────────────────────────────────────────────────

if COLAB_MODE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR   = Path('/content/drive/MyDrive/ZapalloAI')
    SWEET_DIR  = BASE_DIR / 'sweet_pumpkin'
    CUCURB_DIR = BASE_DIR / 'cucurbit_leaf'
    OUTPUT_DIR = BASE_DIR / 'dataset_processed'
else:
    # Detectar raíz del repositorio (funciona desde cualquier subcarpeta)
    NOTEBOOK_DIR = Path(os.path.abspath(''))
    # Sube hasta encontrar la carpeta 'model'
    ROOT = NOTEBOOK_DIR
    for _ in range(5):
        if (ROOT / 'model').exists():
            break
        ROOT = ROOT.parent
    BASE_DIR   = ROOT / 'model' / 'data'
    SWEET_DIR  = BASE_DIR / 'raw' / 'sweet_pumpkin'
    CUCURB_DIR = BASE_DIR / 'raw' / 'cucurbit_leaf'
    OUTPUT_DIR = BASE_DIR / 'processed'

# Clases objetivo
CLASSES = ['healthy', 'downy_mildew', 'leaf_curl', 'mosaic_virus', 'red_beetle']

# Mapeo: nombre de carpeta en disco → clase normalizada
CLASS_MAP = {
    # Sweet Pumpkin Disease Recognition (Dataset 2)
    'Healthy':        'healthy',
    'Powdery Mildew': 'downy_mildew',
    'Leaf Curl':      'leaf_curl',
    'Mosaic Virus':   'mosaic_virus',
    'Red Beetle':     'red_beetle',
    # Cucurbit Leaf Disease Dataset (Dataset 1)
    'healthy':         'healthy',
    'downy_mildew':    'downy_mildew',
    'leaf_curl':       'leaf_curl',
    'leaf_curl_virus': 'leaf_curl',
    'mosaic':          'mosaic_virus',
    'mosaic_virus':    'mosaic_virus',
}

TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f'Modo: {"Google Colab" if COLAB_MODE else "Local"}')
print(f'Sweet Pumpkin : {SWEET_DIR}')
print(f'Cucurbit Leaf : {CUCURB_DIR}')
print(f'Output        : {OUTPUT_DIR}')

# Verificar que existen los datasets
for d, name in [(SWEET_DIR, 'sweet_pumpkin'), (CUCURB_DIR, 'cucurbit_leaf')]:
    if d.exists():
        clases = [x.name for x in d.iterdir() if x.is_dir()]
        print(f'  ✅ {name}: {clases}')
    else:
        print(f'  ❌ {name} NO encontrado en {d}')
        print(f'     → Descarga el dataset y colócalo en esa ruta')

In [ ]:
# ── 2. Detectar y eliminar duplicados con pHash ───────────────────
def get_all_images(base_dir: Path) -> list:
    if not base_dir.exists():
        return []
    exts = ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']
    imgs = []
    for ext in exts:
        imgs.extend(base_dir.rglob(ext))
    return imgs

def compute_hashes(images: list) -> dict:
    hashes = {}
    for i, img_path in enumerate(images):
        if i % 500 == 0 and i > 0:
            print(f'   {i}/{len(images)}...')
        try:
            with Image.open(img_path) as img:
                hashes[str(img_path)] = imagehash.phash(img)
        except Exception:
            pass
    return hashes

print('Recopilando imágenes...')
sweet_imgs  = get_all_images(SWEET_DIR)
cucurb_imgs = get_all_images(CUCURB_DIR)
all_imgs    = sweet_imgs + cucurb_imgs

print(f'Sweet Pumpkin : {len(sweet_imgs):,} imágenes')
print(f'Cucurbit Leaf : {len(cucurb_imgs):,} imágenes')
print(f'Total         : {len(all_imgs):,} imágenes')

if len(all_imgs) == 0:
    print('\n⛔ No se encontraron imágenes. Verifica que los datasets están en las rutas correctas.')
    raise SystemExit(0)

print('\nCalculando hashes para detectar duplicados...')
hashes = compute_hashes(all_imgs)

seen = {}
to_skip = set()
for path, h in hashes.items():
    key = str(h)
    if key in seen:
        to_skip.add(path)
    else:
        seen[key] = path

print(f'\n✅ Duplicados exactos: {len(to_skip)}')
print(f'   Imágenes únicas  : {len(all_imgs) - len(to_skip):,}')

In [ ]:
# ── 3. Crear estructura de salida ─────────────────────────────────
for split in ['train', 'val', 'test']:
    for cls in CLASSES:
        (OUTPUT_DIR / split / cls).mkdir(parents=True, exist_ok=True)

print(f'✅ Estructura creada en: {OUTPUT_DIR}')

In [ ]:
# ── 4. Split estratificado y copia ────────────────────────────────
def process_dataset(src_dir: Path, class_map: dict, prefix: str):
    if not src_dir.exists():
        print(f'  ⚠️ No existe: {src_dir}')
        return

    for cls_dir in sorted(src_dir.iterdir()):
        if not cls_dir.is_dir():
            continue

        dst_cls = class_map.get(cls_dir.name)
        if dst_cls is None:
            print(f'  ⚠️ Sin mapeo: "{cls_dir.name}" — ignorando')
            continue

        imgs = [
            p for p in
            list(cls_dir.glob('*.jpg')) + list(cls_dir.glob('*.jpeg')) +
            list(cls_dir.glob('*.png')) + list(cls_dir.glob('*.JPG'))
            if str(p) not in to_skip
        ]
        random.shuffle(imgs)
        n = len(imgs)
        if n == 0:
            print(f'  ⚠️ {cls_dir.name}: 0 imágenes')
            continue

        n_train = int(n * TRAIN_RATIO)
        n_val   = int(n * VAL_RATIO)

        splits_map = {
            'train': imgs[:n_train],
            'val':   imgs[n_train:n_train + n_val],
            'test':  imgs[n_train + n_val:],
        }

        for split_name, split_imgs in splits_map.items():
            for img_path in split_imgs:
                dst_name = f'{prefix}_{img_path.name}'
                dst_path = OUTPUT_DIR / split_name / dst_cls / dst_name
                if not dst_path.exists():
                    shutil.copy2(img_path, dst_path)

        print(f'  ✅ {cls_dir.name:20s} → {dst_cls:15s}: {n:5d} imgs '
              f'(train={n_train}, val={n_val}, test={n - n_train - n_val})')

print('📂 Procesando Sweet Pumpkin...')
process_dataset(SWEET_DIR, CLASS_MAP, 'swe')

print('\n📂 Procesando Cucurbit Leaf...')
process_dataset(CUCURB_DIR, CLASS_MAP, 'cuc')

print('\n✅ Copia completada')

In [ ]:
# ── 5. Estadísticas del dataset final ────────────────────────────
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'font.family': 'DejaVu Sans'})

final_counts = {split: {} for split in ['train', 'val', 'test']}
for split in ['train', 'val', 'test']:
    for cls in CLASSES:
        p = OUTPUT_DIR / split / cls
        n = sum(1 for _ in p.glob('*.*')) if p.exists() else 0
        final_counts[split][cls] = n

print(f"{'Clase':<20} {'Train':>8} {'Val':>8} {'Test':>8} {'TOTAL':>8}")
print('-' * 52)
grand_total = 0
for cls in CLASSES:
    t  = final_counts['train'].get(cls, 0)
    v  = final_counts['val'].get(cls, 0)
    ts = final_counts['test'].get(cls, 0)
    tot = t + v + ts
    grand_total += tot
    print(f"{cls:<20} {t:>8} {v:>8} {ts:>8} {tot:>8}")
print('-' * 52)
print(f"{'TOTAL':<20} {sum(final_counts['train'].values()):>8} "
      f"{sum(final_counts['val'].values()):>8} "
      f"{sum(final_counts['test'].values()):>8} "
      f"{grand_total:>8}")

# Balance de clases
train_counts = [final_counts['train'].get(c, 0) for c in CLASSES]
max_c = max(train_counts) if train_counts else 1
min_c = min(c for c in train_counts if c > 0) if any(train_counts) else 1
ratio = max_c / min_c
print(f'\nBalance train max/min: {ratio:.2f}x', end=' ')
print('✅ OK' if ratio <= 3 else '⚠️ Desbalanceado — se aplicará augmentation')

# Gráfico
colors = ['#2D6A4F', '#52B788', '#74C69D', '#95D5B2', '#D8F3DC']
x = range(len(CLASSES))
t_vals  = [final_counts['train'].get(c, 0) for c in CLASSES]
v_vals  = [final_counts['val'].get(c, 0) for c in CLASSES]
ts_vals = [final_counts['test'].get(c, 0) for c in CLASSES]

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x, t_vals,  label='Train', color='#2D6A4F')
ax.bar(x, v_vals,  bottom=t_vals, label='Val', color='#52B788')
ax.bar(x, ts_vals, bottom=[t+v for t,v in zip(t_vals,v_vals)],
       label='Test', color='#95D5B2')
ax.set_xticks(list(x))
ax.set_xticklabels(CLASSES, rotation=15, ha='right')
ax.set_ylabel('Imágenes')
ax.set_title('Distribución del Dataset ZapalloAI — Procesado')
ax.legend()
plt.tight_layout()
out_fig = OUTPUT_DIR.parent / 'dataset_distribution.png'
plt.savefig(out_fig, dpi=150)
plt.show()
print(f'Gráfico guardado: {out_fig}')

In [ ]:
# ── 6. Augmentation offline para clases minoritarias ─────────────
import albumentations as A
import cv2

augment_pipeline = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.Rotate(limit=45, p=0.7),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.6),
    A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=40, val_shift_limit=20, p=0.5),
    A.GaussianBlur(blur_limit=(3, 7), p=0.2),
    A.GaussNoise(var_limit=(10.0, 50.0), p=0.2),
    A.CoarseDropout(num_holes_range=(1, 8), hole_height_range=(10, 30),
                    hole_width_range=(10, 30), p=0.3),
])

TARGET_PER_CLASS = 1000  # Objetivo mínimo en train

aug_total = 0
for cls in CLASSES:
    train_dir = OUTPUT_DIR / 'train' / cls
    if not train_dir.exists():
        continue
    existing = list(train_dir.glob('*.jpg')) + list(train_dir.glob('*.png'))
    current  = len(existing)
    needed   = max(0, TARGET_PER_CLASS - current)

    if needed == 0:
        print(f'  OK  {cls}: {current} imgs (>= {TARGET_PER_CLASS})')
        continue

    print(f'  AUG {cls}: {current} → +{needed} imágenes...')
    gen = 0
    while gen < needed:
        src = random.choice(existing)
        img_bgr = cv2.imread(str(src))
        if img_bgr is None:
            continue
        img_rgb  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        aug_rgb  = augment_pipeline(image=img_rgb)['image']
        aug_bgr  = cv2.cvtColor(aug_rgb, cv2.COLOR_RGB2BGR)
        out_path = train_dir / f'aug_{cls}_{gen:05d}.jpg'
        cv2.imwrite(str(out_path), aug_bgr, [cv2.IMWRITE_JPEG_QUALITY, 90])
        gen += 1
    aug_total += gen

print(f'\n✅ Augmentation: +{aug_total} imágenes generadas')

In [ ]:
# ── 7. Guardar dataset.yaml para entrenamiento ───────────────────
yaml_path = OUTPUT_DIR.parent / 'dataset_final.yaml'

yaml_content = f"""# Dataset ZapalloAI — generado por Notebook 02
path: {OUTPUT_DIR.resolve()}
train: train
val: val
test: test

nc: 5
names:
  0: healthy
  1: downy_mildew
  2: leaf_curl
  3: mosaic_virus
  4: red_beetle
"""

with open(yaml_path, 'w', encoding='utf-8') as f:
    f.write(yaml_content)

# Contar totales finales
total_final = sum(
    sum(1 for _ in (OUTPUT_DIR / split / cls).glob('*.*'))
    for split in ['train', 'val', 'test']
    for cls in CLASSES
    if (OUTPUT_DIR / split / cls).exists()
)

print(f'✅ dataset_final.yaml guardado: {yaml_path}')
print(f'📊 Total imágenes en dataset procesado: {total_final:,}')
print(f'\n🚀 SIGUIENTE: Ejecutar Notebook 03 con:')
print(f'   data = "{OUTPUT_DIR.resolve()}"')